A notebook for visualizing the results of the script fit_one_dim_synthesis_example

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import copy
import os
import pathlib

import matplotlib
from matplotlib import cm
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import torch

from janelia_core.math.basic_functions import list_grid_pts
from janelia_core.ml.fitting import match_torch_module
from janelia_core.math.basic_functions import bound
from janelia_core.math.basic_functions import pts_in_arc
from janelia_core.ml.utils import list_torch_devices
from janelia_core.stats.regression import corr
from janelia_core.stats.regression import r_squared

from probabilistic_model_synthesis.gaussian_nonlinear_regression import approximate_elbo
from probabilistic_model_synthesis.gaussian_nonlinear_regression import eval_check_point_perf
from probabilistic_model_synthesis.gaussian_nonlinear_regression import Fitter
from probabilistic_model_synthesis.gaussian_nonlinear_regression import load_check_points
from probabilistic_model_synthesis.gaussian_nonlinear_regression import predict
from probabilistic_model_synthesis.gaussian_nonlinear_regression import PosteriorCollection
from probabilistic_model_synthesis.gaussian_nonlinear_regression import PriorCollection
from probabilistic_model_synthesis.gaussian_nonlinear_regression import VICollection
from probabilistic_model_synthesis.simulation import efficient_cone_and_projected_interval_sample
from probabilistic_model_synthesis.visualization import import_style
import_style()

In [ ]:
# %matplotlib notebook

## Parameters go here 

In [ ]:
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / 'probabilistic_model_synthesis').is_dir() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

# Location of folder with the results we should visualize
rs_folder = REPO_ROOT / 'results/publication_results/gnlr/simulation'

# File holding fitting results
rs_file = 'full_simulations.pt'

# Sub-folders holding check points
combined_cp_folder = 'comb_cps'
single_cp_folder = 'single_cps'

# Type of results we should look at - combined or single
rs_type = 'single'

# True if we should look at fitting logs
vis_fit_logs = True

# Index of the example system we should visualize results for
ex_s_i = 2 # 1 is a good example

# Number of samples we use for validation data when performing early stopping
n_validation_smps = 1000

# Number of samples we generate for evaluating model performance
n_eval_smps = 10000

# Location we should save plots to - if this folder doesn't exist we will create it 
save_loc = rs_folder

## Determine if GPU is availalbe 

In [ ]:
compute_devices, gpu_available = list_torch_devices()
if gpu_available:
    compute_device = compute_devices[0]
else:
    compute_device = torch.device('cpu')

## Load results and prepare what we need for later analysis

In [ ]:
rs_file_path = pathlib.Path(rs_folder) / rs_file

In [ ]:
rs = torch.load(rs_file_path)

Pull out constants, paths, etc we will need below

In [ ]:
n_ex_systems = rs['ps']['n_ex_systems']

subj_props = rs['ind_props']
fit_ps = rs['ps']
single_fit_inds = rs['ps']['single_fit_inds']
n_single_fit_systems = len(single_fit_inds)

combined_cp_folder = pathlib.Path(rs_folder) / combined_cp_folder
single_cp_folder = pathlib.Path(rs_folder) / single_cp_folder

cmap = plt.get_cmap('inferno')

## Apply retroactive early stopping to synthesized models and for model fit individually

Generate validation data for each subject 

In [ ]:
eval_data = [None]*n_ex_systems
for s_i in range(n_ex_systems):
    
    # We generate data with the same active neurons and behevioral range as was in the training data for
    # this subject
    with torch.no_grad():
        x_i_validation = efficient_cone_and_projected_interval_sample(n_smps=n_validation_smps,
                                                                        locs=rs['ind_props'][s_i],
                                                                        ctr=torch.tensor([.5, .5]),
                                                                        ang_range=rs['ang_ranges'][s_i],
                                                                        w=rs['ind_true_mdls'][s_i].w.detach(),
                                                                        interval=rs['ind_intervals'][s_i],
                                                                        big_std=1.0,
                                                                        small_std=0,
                                                                        device=compute_device)
    
        # Before generating y data, move model to GPU (if possible)
        rs['ind_true_mdls'][s_i].to(compute_device)
        y_i_validation = rs['ind_true_mdls'][s_i].sample(x=x_i_validation)
        
        # Move data back to cpu to save GPU memory
        x_i_validation = x_i_validation.to('cpu')
        y_i_validation = y_i_validation.to('cpu')
        
        eval_data[s_i] = (x_i_validation, y_i_validation)
    

Evaluate check point performance for the synthesized models

In [ ]:
comb_sp_cps, comb_sp_cp_epochs = load_check_points(cp_dir=combined_cp_folder, cp_str='cp_sp')
comb_ip_cps, comb_ip_cp_epochs = load_check_points(cp_dir=combined_cp_folder, cp_str='cp_ip')

comb_sp_perf = eval_check_point_perf(cps=comb_sp_cps, eval_data=eval_data, subj_props=subj_props, 
                                     eval_device=compute_device)

comb_ip_perf = eval_check_point_perf(cps=comb_ip_cps, eval_data=eval_data, subj_props=subj_props, 
                                     eval_device=compute_device)

Define a smaller helper function for plotting check point performance 

In [ ]:
def plot_cp_perf(plt_epochs, perf_vls, ax):
    
    n_subjs = perf_vls.shape[1]
    for s_i in range(n_subjs):
        ax.plot(plt_epochs, perf_vls[:, s_i])
    
    if n_subjs > 1:
        ax.plot(plt_epochs, np.mean(perf_vls, axis=1), 'k-', linewidth=4)

Plot performance across check points

In [ ]:
plt.figure()
ax = plt.subplot(1,2,1)
plot_cp_perf(comb_sp_cp_epochs, comb_sp_perf, ax)
plt.ylim([-1, 1])
plt.title('Combined SP CP Perf')

ax = plt.subplot(1,2,2)
plot_cp_perf(comb_ip_cp_epochs, comb_ip_perf, ax)
plt.ylim([-1, 1])
plt.title('Combined IP CP Perf')


Pick the best check point for the synthesized models

We pick only check points for models with individual posteriors (as we consider fitting with shared posteriors to be an initiliazation step).  We pick one check point for all subjects - as all models are synthesized together. 

In [ ]:
best_comb_ip_cp_ind = np.argmax(np.mean(comb_ip_perf, axis=1))

comb_vi_colls = [VICollection.from_checkpoint(comb_ip_cps[best_comb_ip_cp_ind]['vi_collections'][s_i])
                 for s_i in range(n_ex_systems)]

comb_priors = PriorCollection.from_checkpoint(comb_ip_cps[best_comb_ip_cp_ind]['priors'])

Evaluate performance for the models fit to individual systems

In [ ]:
single_cp_perf = [None]*n_single_fit_systems
for i, s_i in enumerate(single_fit_inds):
    
    cp_folder_i = single_cp_folder / ('s_' + str(s_i))


    single_sp_cps_i, single_sp_cp_epochs_i = load_check_points(cp_dir=cp_folder_i, cp_str='cp_sp')
    single_ip_cps_i, single_ip_cp_epochs_i = load_check_points(cp_dir=cp_folder_i, cp_str='cp_ip')

    single_sp_perf_i = eval_check_point_perf(cps=single_sp_cps_i, eval_data=[eval_data[s_i]], 
                                             subj_props=[subj_props[s_i]], 
                                             eval_device=compute_device)

    single_ip_perf_i = eval_check_point_perf(cps=single_ip_cps_i, eval_data=[eval_data[s_i]], 
                                             subj_props=[subj_props[s_i]], 
                                             eval_device=compute_device)
    
    single_cp_perf[i] = {'sp_epochs': single_sp_cp_epochs_i, 'sp_perf': single_sp_perf_i, 
                           'ip_epochs': single_ip_cp_epochs_i, 'ip_perf': single_ip_perf_i, 
                            'sp_cps': single_sp_cps_i, 'ip_cps': single_ip_cps_i}

Plot performance across check points

In [ ]:
plt.figure()
ax = plt.subplot(1,2,1)
for i in range(n_single_fit_systems):
    plot_cp_perf(single_cp_perf[i]['sp_epochs'], single_cp_perf[i]['sp_perf'], ax)
plt.ylim([-1, 1])
plt.title('Individual SP CP Perf')

ax = plt.subplot(1,2,2)
for i in range(n_ex_systems):
    plot_cp_perf(single_cp_perf[i]['ip_epochs'], single_cp_perf[i]['ip_perf'], ax)
plt.ylim([-1, 1])
plt.title('Individual IP CP Perf')

Pick the best check point for each model fit to an example system in isolation 

In [ ]:
single_fits = [None]*n_single_fit_systems
for i in range(n_single_fit_systems):
    best_cp_ind = np.argmax(single_cp_perf[i]['ip_perf'])
    
    single_fits[i] = {'vi_coll': VICollection.from_checkpoint(
         single_cp_perf[i]['ip_cps'][best_cp_ind]['vi_collections'][0]), 
                      'priors': PriorCollection.from_checkpoint(single_cp_perf[i]['ip_cps'][best_cp_ind]['priors'])}


## Pick key quantities we need when showing results below

In [ ]:
single_ex_i = np.argwhere(np.asarray(single_fit_inds) == ex_s_i).item()

In [ ]:
# Determine the domain of the shared m function
min_interval = np.min(np.asarray(rs['ind_intervals']))
max_interval = np.max(np.asarray(rs['ind_intervals']))
min_l_range = min_interval*fit_ps['s_in']
max_l_range = max_interval*fit_ps['s_in']

# Pull out neurons, their properties and true weights for the example system we use for visualization 
ex_active_neurons = np.asarray((torch.var(rs['ind_data'][ex_s_i][0], dim=0) > 1E-10).tolist())
ex_neuron_props = rs['ind_props'][ex_s_i]
ex_true_neuron_weights = rs['ind_true_mdls'][ex_s_i].w.detach().cpu().numpy()

# Pull out true CPD
true_priors = rs['true_priors']

if rs_type == 'combined':
    # The fit logs we plot
    fit_logs = rs['comb_fit_rs']['ip']['logs']
    # The range the shared function should be learnable over 
    m_fit_range = torch.tensor([[min_l_range], [max_l_range]])
    # The fit shared function
    m_fit = comb_vi_colls[0].mdl.m
    # The posterior mean over neuron weights for the example system
    ex_w_fit_mn = comb_vi_colls[ex_s_i].posteriors.w_post(ex_neuron_props).detach().cpu().numpy()
    # The fit CPD
    fit_priors = comb_priors

elif rs_type == 'single':
    # The fit logs we plot
    fit_logs = rs['single_fit_rs'][single_ex_i]['ip']['logs']
    # The range the shared function should be learnable over 
    m_fit_range = torch.tensor([[rs['ind_intervals'][ex_s_i][0]], [rs['ind_intervals'][ex_s_i][1]]], dtype=torch.float32)*fit_ps['s_in']
    # The fit shared function
    m_fit = single_fits[single_ex_i]['vi_coll'].mdl.m
    # The posterior mean over neuron weights for the example system
    ex_w_fit_mn = single_fits[single_ex_i]['vi_coll'].posteriors.w_post(ex_neuron_props).detach().cpu().numpy()
    # The fit CPD
    fit_priors = single_fits[single_ex_i]['priors']

## Examine fitting logs, if requested 

In [ ]:
if vis_fit_logs:
    for log in fit_logs:
        Fitter.plot_log(log)

## Create folder to save results into if needed

In [ ]:
save_folder = pathlib.Path(save_loc)
type_save_loc = save_folder / (rs_type + '_images')
if not os.path.isdir(save_loc):
    print('Creating folder to save plots into: ' + str(save_folder))
    os.makedirs(save_folder)
    
if not os.path.isdir(type_save_loc):
    print('Creating folder to save plots into: ' + str(type_save_loc))
    os.makedirs(type_save_loc)

## Define some helper functions

In [ ]:
# Helper formatting function
def format_box(ax):
    ax.set_aspect('equal', 'box')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.xlabel('property dim 1 (a.u.)')
    plt.ylabel('property dim 2 (a.u.)')

In [ ]:
# Helper function for evaluating model performance

def eval_mdl_perf(eval_x: torch.Tensor, eval_y: torch.Tensor, coll, priors, n_elbo_smps=100):

    with torch.no_grad():
        y_hat = predict(coll=coll, x=eval_x, sample=False).cpu().numpy()

    # Measure performance of the predictions
    eval_r_sq = r_squared(eval_y.cpu().numpy(), y_hat) 

    
    eval_corr = corr(eval_y.cpu().numpy(), y_hat) 
    
    # Measure the ELBO for the fit models on the data in the training domain
    orig_data = coll.data
    coll.data = [eval_x, eval_y]
    with torch.no_grad():
        eval_elbo = approximate_elbo(coll=coll, priors=priors, n_smps=n_elbo_smps, 
                                     skip_s_in_kl=True, skip_b_in_kl=True, skip_s_out_kl=True, 
                                     skip_b_out_kl=True)
    
    return {'r_sq': eval_r_sq, 'corr': eval_corr, 'elbo': eval_elbo['elbo'].cpu().numpy(), 
            'decomposed_elbo': eval_elbo, 'y_true': eval_y.cpu().numpy(), 'y_hat': y_hat}
    

In [ ]:
def vis_perf(eval_rs: dict, ax, metric='r_sq', min_vl=None, min_plot_vl = None, max_plot_vl = None, 
             title=None, ex_i = None):
    """ 
    Plots relative performance of models fit to combined data vs individual data. 
    
    In particular makes scatter plots of performance comparing what happens when we fit 
    models together with DPMS vs. fitting to individual example systems alone
    """
    
    comb_train_domain_vls = np.asarray([rs['comb_train_domain_perf'][metric].item() for rs in eval_rs])
    sing_train_domain_vls = np.asarray([rs['sing_train_domain_perf'][metric].item() for rs in eval_rs])
    
    comb_full_vls = np.asarray([rs['comb_full_perf'][metric].item() for rs in eval_rs])
    sing_full_vls = np.asarray([rs['sing_full_perf'][metric].item() for rs in eval_rs])
    
    # Enforce min value if we are suppose to
    if min_vl is not None:
        comb_train_domain_vls = bound(comb_train_domain_vls, min_vl, np.inf)
        sing_train_domain_vls = bound(sing_train_domain_vls, min_vl, np.inf)
        comb_full_vls = bound(comb_full_vls, min_vl, np.inf)
        sing_full_vls = bound(sing_full_vls, min_vl, np.inf)

    # Generate plot
    all_vls = np.concatenate([comb_train_domain_vls, sing_train_domain_vls, comb_full_vls, sing_full_vls])
    if min_plot_vl is None:
        min_plot_vl = np.min(all_vls)
    if max_plot_vl is None:
        max_plot_vl= np.max(all_vls)
    
    ax.plot([min_plot_vl, max_plot_vl], [min_plot_vl, max_plot_vl], 'k--', label='_nolegend_')
    ax.plot(sing_train_domain_vls, comb_train_domain_vls, 'k.',)
    ax.plot(sing_full_vls, comb_full_vls, '.', color='gray')
    
    if ex_i is not None:
        ax.plot(sing_train_domain_vls[ex_i], comb_train_domain_vls[ex_i], 'ko', markerfacecolor='none',
                markersize=15)
        ax.plot(sing_full_vls[ex_i], comb_full_vls[ex_i], 'o', markerfacecolor='none',
                markersize=15, color='gray')
    
    plt.legend(['within training Distribution', 'out of training distribution'])
    if title is not None:
        plt.title(title)
    ax.set_aspect('equal', 'box')
    
    def _summary_str(name, vls):
        n = len(vls)
        mean = np.mean(vls)
        sem = np.std(vls, ddof=1) / np.sqrt(n) if n > 1 else np.nan
        return f'{name}: mean={mean:.4g}, SEM={sem:.4g}, n={n}'
    
    print(_summary_str('within training, DPMS', comb_train_domain_vls))
    print(_summary_str('within training, no DPMS', sing_train_domain_vls))
    print(_summary_str('out of training, DPMS', comb_full_vls))
    print(_summary_str('out of training, no DPMS', sing_full_vls))
    

In [ ]:
def get_fit_to_true_cpd_scale(rs_type, fit_priors, true_priors, fit_pts, ang_range=None):
    """
    Learns the best scale to match the fit CPD to the true CPD. 
    
    """

    if rs_type == 'combined':
        sc_grid_pts = fit_pts
    elif rs_type == 'single':   
        sc_grid_pts, _ = list_grid_pts(grid_limits=np.asarray([[0, 1.0], [0, 1.0]]), n_pts_per_dim=[100,100])
        sc_grid_pts = torch.tensor(sc_grid_pts[pts_in_arc(sc_grid_pts, np.asarray([.5, .5]), ang_range),:])

    return  np.linalg.lstsq(fit_priors.w_prior(sc_grid_pts).detach().numpy(),
                            true_priors.w_prior(sc_grid_pts).detach().numpy(), rcond=None)[0].item()


## Visualize the active neurons for one example system

In [ ]:
w_vis_min = np.min(ex_true_neuron_weights)
w_vis_max = np.max(ex_true_neuron_weights)

# Get colormap
norm = mcolors.CenteredNorm(vcenter=0, halfrange=None)  
cmap = plt.get_cmap('coolwarm')
colors = cmap(norm(ex_true_neuron_weights)).squeeze()

# Make inactive neurons transparent
colors[~ex_active_neurons, 3] = 1

In [ ]:
sub_smp_int = 5

fig = plt.figure(figsize=(1.8, 1.4))
ax = plt.subplot(1,1,1)

im = ax.scatter(ex_neuron_props[::sub_smp_int, 0],
                ex_neuron_props[::sub_smp_int, 1],
                color=colors[::sub_smp_int],
                s=1,
                marker='.', 
                rasterized=False)

# Need a ScalarMappable for the colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label(label='Weight (a.u.)', fontsize=8, labelpad=-10)
cbar.set_ticks([-4.5, 4.5])

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_aspect('equal')
ax.set_xlabel('Prop. dim 1 (a.u.)', fontsize=8, labelpad=-2)
ax.set_ylabel('Prop. dim 2 (a.u.)', fontsize=8, labelpad=-2)

# fig.subplots_adjust(wspace=-.75, hspace=-0.4, left=0.00, right=0.99, top=0.99, bottom=0.0)
fig.savefig(save_folder / ('active_neurons_s_' + str(ex_s_i) + '.svg'), format='svg', transparent=True)

## Learn the scale that minimizes error between true and fit CPD

In [ ]:
pts, dim_pts = list_grid_pts(grid_limits=np.asarray([[0, 1.0], [0, 1.0]]), n_pts_per_dim=[100,100])
pts = torch.tensor(pts, dtype=torch.float)

In [ ]:
true_to_fit_cpd_sc = get_fit_to_true_cpd_scale(rs_type, fit_priors, true_priors, fit_pts=pts, 
                                               ang_range=rs['ang_ranges'][ex_s_i])

## Visualize the true and fit shared function 

In [ ]:
m_true = rs['m_true']

l_plot_pts = np.linspace(min_l_range,max_l_range,1000)
m_true_pts = m_true(torch.tensor(l_plot_pts)).numpy()

In [ ]:
m_fit_cp = copy.deepcopy(m_fit)
m_fit_scaled = torch.nn.Sequential(torch.nn.Linear(1, 1, bias=False), m_fit_cp)
m_fit_scaled[0].weight.data[0] = 1/true_to_fit_cpd_sc
m_fit_scaled = m_fit_scaled.cpu()

Plot the scaled and offset fit shared function

In [ ]:
ex_int = np.asarray(rs['ind_intervals'][ex_s_i])/np.sqrt(rs['ps']['n_input_var_range'][0])
ex_int_w = ex_int[1] - ex_int[0]

In [ ]:
m_fit_pts = m_fit_scaled(torch.tensor(l_plot_pts)).detach().numpy()

In [ ]:
plt.figure(figsize=(1.8, 1.5))
ax = plt.subplot(1,1,1)
ax.add_patch(Rectangle([ex_int[0], -2], ex_int_w, 3.0, color='lightgrey'))
ax.plot(l_plot_pts, m_true_pts, color='fuchsia', linestyle='-', linewidth=2)
ax.plot(l_plot_pts, m_fit_pts, color='k', linestyle='-', linewidth=1)
ax.set_ylim(-2, 2)

ax.set_xticks([-2, 0, 2])
ax.set_yticks([-1.5, 0, 1.5])

ax.spines[['right', 'top']].set_visible(False)

plt.savefig(type_save_loc / f'm_fit_{rs_type}.svg', format='svg')

## Visualize the fit mean of the posterior over weights for the example subject 

In [ ]:
w_fit_mn_scaled = true_to_fit_cpd_sc*ex_w_fit_mn

w_vis_min = np.min(w_fit_mn_scaled)
w_vis_max = np.max(w_fit_mn_scaled)

# Get colormap
norm = mcolors.CenteredNorm(vcenter=0, halfrange=None)  
cmap = plt.get_cmap('coolwarm')
colors = cmap(norm(w_fit_mn_scaled)).squeeze()

In [ ]:
sub_smp_int = 5

fig = plt.figure(figsize=(2.2, 1.7))
ax = plt.subplot(1,1,1)

im = ax.scatter(ex_neuron_props[::sub_smp_int, 0],
                ex_neuron_props[::sub_smp_int, 1],
                color=colors[::sub_smp_int],
                s=1,
                marker='.', 
                rasterized=False)

# Need a ScalarMappable for the colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label(label='Weight (a.u.)', fontsize=8, labelpad=-10)
cbar.set_ticks([-4.5, 4.5])

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_aspect('equal')
ax.set_xlabel('Prop. dim 1 (a.u.)', fontsize=8, labelpad=-5)
ax.set_ylabel('Prop. dim 2 (a.u.)', fontsize=8, labelpad=-5)

format_box(ax)
plt.savefig(type_save_loc / ('posterior_weights_s_' + str(ex_s_i) + f'_{rs_type}.svg'), format='svg')

## Visualize the mean and standard deviation of the true CPD 

In [ ]:
true_w_mn = true_priors.w_prior(pts).detach().numpy()
true_w_std = np.concatenate([d.std_f(pts).detach().cpu().numpy() for d in true_priors.w_prior.dists], axis=1)
true_w_mn_im = true_w_mn.reshape([100,100]).transpose()
true_w_std_im = true_w_std.reshape([100,100]).transpose()

fig = plt.figure(figsize=(1.8, 1.4))
ax = plt.subplot(1,1,1)
im = ax.imshow(true_w_mn_im, origin='lower', extent=[0, 1.0, 0, 1.0], cmap=cmap, norm=mcolors.CenteredNorm(vcenter=0, halfrange=None))
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(label='Weight (a.u.)', fontsize=8, labelpad=-10)
cbar.set_ticks([-4.5, 4.5])

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_aspect('equal')
ax.set_xlabel('Prop. dim 1 (a.u.)', fontsize=8, labelpad=-2)
ax.set_ylabel('Prop. dim 2 (a.u.)', fontsize=8, labelpad=-2)
format_box(ax)
plt.title('True CPD Mean')
plt.savefig(save_folder / 'true_cpd_mean.svg', format='svg')

fig = plt.figure(figsize=(1.8, 1.4))
ax = plt.subplot(1,1,1)
im = ax.imshow(true_w_std_im, origin='lower', extent=[0, 1.0, 0, 1.0], cmap=cmap, vmax=.4)
std_vmin, std_vmax = im.get_clim() # Keep track of color limits
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(label='Weight (a.u.)', fontsize=8, labelpad=-10)
cbar.set_ticks([0.05, 0.35])

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_aspect('equal')
ax.set_xlabel('Prop. dim 1 (a.u.)', fontsize=8, labelpad=-2)
ax.set_ylabel('Prop. dim 2 (a.u.)', fontsize=8, labelpad=-2)
format_box(ax)
plt.title('True CPD Standard Deviation')
plt.savefig(save_folder / 'true_cpd_std.svg', format='svg')

## Visualize the mean and standard deviation of the fit CPD

Sample fit CPD in a grid for visualization

In [ ]:
fit_w_mn = fit_priors.w_prior(pts).detach().numpy()
fit_w_std = np.concatenate([d.std_f(pts).detach().cpu().numpy() for d in fit_priors.w_prior.dists], axis=1)

In [ ]:
fit_w_mn_scaled = fit_w_mn*true_to_fit_cpd_sc
fit_w_std_scaled = fit_w_std*np.abs(true_to_fit_cpd_sc)

fit_w_mn_scaled_im = fit_w_mn_scaled.reshape([100,100]).transpose()
fit_w_std_scaled_im = fit_w_std_scaled.reshape([100,100]).transpose()

In [ ]:
fig = plt.figure(figsize=(1.8, 1.4))
ax = plt.subplot(1,1,1)
im = ax.imshow(fit_w_mn_scaled_im, origin='lower', extent=[0, 1.0, 0, 1.0], cmap=cmap, norm=mcolors.CenteredNorm(vcenter=0, halfrange=None))
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(label='Weight (a.u.)', fontsize=8, labelpad=-10)
cbar.set_ticks([-4.5, 4.5])

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_aspect('equal')
ax.set_xlabel('Prop. dim 1 (a.u.)', fontsize=8, labelpad=-2)
ax.set_ylabel('Prop. dim 2 (a.u.)', fontsize=8, labelpad=-2)
format_box(ax)
plt.title('Fit CPD Mean')
plt.savefig(type_save_loc / 'fit_cpd_mean.svg', format='svg')

fig = plt.figure(figsize=(1.8, 1.4))
ax = plt.subplot(1,1,1)
im = ax.imshow(fit_w_std_scaled_im, origin='lower', extent=[0, 1.0, 0, 1.0], cmap=cmap, vmax=.4)
std_vmin, std_vmax = im.get_clim() # Keep track of color limits
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(label='Weight (a.u.)', fontsize=8, labelpad=-10)
cbar.set_ticks([0.05, 0.35])

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_aspect('equal')
ax.set_xlabel('Prop. dim 1 (a.u.)', fontsize=8, labelpad=-2)
ax.set_ylabel('Prop. dim 2 (a.u.)', fontsize=8, labelpad=-2)
format_box(ax)
plt.title('Fit CPD Standard Deviation')
plt.savefig(type_save_loc / 'fit_cpd_std.svg', format='svg')

## Make a scatter plot of true vs fit mean for example subject 

Before plotting we first scale the fit weights to account for the non-identifiability in scale

In [ ]:
plt.figure(figsize=(1.8, 1.5))
ax = plt.subplot(1,1,1)

pts = np.concatenate([ex_true_neuron_weights, true_to_fit_cpd_sc*ex_w_fit_mn], axis=1)
clrs = np.zeros([pts.shape[0], 3])
clrs[~ex_active_neurons,:] = .8
    
# Here we permute the plotting order of the points so we don't create
# visualization artefacts of regions that look like they are all black or red.
rand_order = np.random.permutation(pts.shape[0])

plt.xlabel('True W')
plt.ylabel('W Post Mn')
ax.scatter(pts[rand_order,0], pts[rand_order,1], marker='.', color=clrs[rand_order,:], s=1)
ax.set_aspect('equal')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_xlim(-5, 5)
ax.set_xticks([-4, 0, 4])
ax.set_yticks([-4, 0, 4])

# Save the figure - use jpg due to large number of vector objects a eps/pdf would create
plt.savefig(type_save_loc / ('true_vs_est_w_s_' + str(ex_s_i) + '.svg'), format='svg', transparent=True)

## Evaluate model performance on test data for models fit to each example system together and individually

Evaluate model performance here

In [ ]:
eval_rs = [None]*n_ex_systems
for i, s_i in enumerate(single_fit_inds):
    
    # ============================================================================================================
    # Generate data using the ground truth example system in two ways:
    #    1) The first is using the same active neurons and behavior range that was observed in the training data
    #    2 The second is using all neurons and all behavioral ranges
    
    # Here we generate data with the same active neurons and behevioral range as was in the training data for
    # this subject
    with torch.no_grad():
        x_i_train_domain = efficient_cone_and_projected_interval_sample(n_smps=n_eval_smps,
                                                                        locs=rs['ind_props'][s_i],
                                                                        ctr=torch.tensor([.5, .5]),
                                                                        ang_range=rs['ang_ranges'][s_i],
                                                                        w=rs['ind_true_mdls'][s_i].w.detach(),
                                                                        interval=rs['ind_intervals'][s_i],
                                                                        big_std=1.0,
                                                                        small_std=0,
                                                                        device=compute_device)
    
        # Before generating y data, move model to GPU (if possible)
        rs['ind_true_mdls'][s_i].to(compute_device)
        y_i_train_domain = rs['ind_true_mdls'][s_i].forward(x=x_i_train_domain)
    
        # Here we generate data when all neurons are active and for the full behavioral range
        x_i_full = efficient_cone_and_projected_interval_sample(n_smps=n_eval_smps,
                                                                locs=rs['ind_props'][s_i],
                                                                ctr=torch.tensor([.5, .5]),
                                                                ang_range=[0, 2*np.pi],
                                                                w=rs['ind_true_mdls'][s_i].w.detach(),
                                                                interval=[min_interval, max_interval],
                                                                big_std=1.0,
                                                                small_std=0,
                                                                device=compute_device)
        
        y_i_full = rs['ind_true_mdls'][s_i].forward(x=x_i_full)
        
        # Move true model back to cpu
        rs['ind_true_mdls'][s_i].to('cpu')
        
    # ============================================================================================================
    # Now we evaluate performance of the models fit collectively and individually 
    sing_i = np.argwhere(np.asarray(single_fit_inds) == s_i).item()
    
    comb_coll = comb_vi_colls[s_i]
    comb_coll.props = rs['ind_props'][s_i]
    comb_priors = comb_priors
    
    sing_coll = single_fits[sing_i]['vi_coll']
    sing_coll.props = rs['ind_props'][s_i]
    sing_priors = single_fits[sing_i]['priors']

    # Move the collections and priors to GPU (if possible) 
    comb_coll.to(compute_device)
    sing_coll.to(compute_device)
    comb_priors.to(compute_device)
    sing_priors.to(compute_device)
    
    # Evaluate model performance
    comb_train_domain_perf = eval_mdl_perf(x_i_train_domain, y_i_train_domain, comb_coll, comb_priors)
    sing_train_domain_perf = eval_mdl_perf(x_i_train_domain, y_i_train_domain, sing_coll, sing_priors)
    
    comb_full_perf = eval_mdl_perf(x_i_full, y_i_full, comb_coll, comb_priors)
    sing_full_perf = eval_mdl_perf(x_i_full, y_i_full, sing_coll, sing_priors)
    
    eval_rs[i] = {'s_i': s_i, 
                  'comb_train_domain_perf': comb_train_domain_perf,
                  'sing_train_domain_perf': sing_train_domain_perf,
                  'comb_full_perf': comb_full_perf, 
                  'sing_full_perf': sing_full_perf}
    
    # Move everything back to cpu
    comb_coll.to('cpu')
    sing_coll.to('cpu')
    comb_priors.to('cpu')
    sing_priors.to('cpu')
    
                  
    print('Done evaluating model performance for subject ' + str(s_i) + '.')
    

## Visualize model performance 

In [ ]:
def vis_perf(eval_rs: dict, ax, metric='r_sq', min_vl=None, min_plot_vl = None, max_plot_vl = None, 
             title=None, ex_i = None):
    """ 
    Plots relative performance of models fit to combined data vs individual data. 
    
    In particular makes scatter plots of performance comparing what happens when we fit 
    models together with DPMS vs. fitting to individual example systems alone
    """
    
    comb_train_domain_vls = np.asarray([rs['comb_train_domain_perf'][metric].item() for rs in eval_rs])
    sing_train_domain_vls = np.asarray([rs['sing_train_domain_perf'][metric].item() for rs in eval_rs])
    
    comb_full_vls = np.asarray([rs['comb_full_perf'][metric].item() for rs in eval_rs])
    sing_full_vls = np.asarray([rs['sing_full_perf'][metric].item() for rs in eval_rs])
    
    # Enforce min value if we are suppose to
    if min_vl is not None:
        comb_train_domain_vls = bound(comb_train_domain_vls, min_vl, np.inf)
        sing_train_domain_vls = bound(sing_train_domain_vls, min_vl, np.inf)
        comb_full_vls = bound(comb_full_vls, min_vl, np.inf)
        sing_full_vls = bound(sing_full_vls, min_vl, np.inf)

    # Generate plot
    all_vls = np.concatenate([comb_train_domain_vls, sing_train_domain_vls, comb_full_vls, sing_full_vls])
    if min_plot_vl is None:
        min_plot_vl = np.min(all_vls)
    if max_plot_vl is None:
        max_plot_vl= np.max(all_vls)


    plt.figure(figsize=(2.2, 2))
    ax = plt.subplot(1,1,1)
    
    ax.plot([min_plot_vl, max_plot_vl], [min_plot_vl, max_plot_vl], 'k--', label='_nolegend_')
    ax.plot(sing_train_domain_vls, comb_train_domain_vls, 'k.',)
    ax.plot(sing_full_vls, comb_full_vls, '.', color='gray')
    
    if ex_i is not None:
        ax.plot(sing_train_domain_vls[ex_i], comb_train_domain_vls[ex_i], 'ko', markerfacecolor='none',
                markersize=5)
        ax.plot(sing_full_vls[ex_i], comb_full_vls[ex_i], 'o', markerfacecolor='none',
                markersize=5, color='gray')
    
    plt.legend(['within training Distribution', 'out of training distribution'])
    if title is not None:
        plt.title(title)
    ax.set_aspect('equal', 'box')

    def _summary_str(name, vls):
        n = len(vls)
        mean = np.mean(vls)
        sem = np.std(vls, ddof=1) / np.sqrt(n) if n > 1 else np.nan
        return f'{name}: mean={mean:.4g}, SEM={sem:.4g}, n={n}'
    
    print(_summary_str('within training, DPMS', comb_train_domain_vls))
    print(_summary_str('within training, no DPMS', sing_train_domain_vls))
    print(_summary_str('out of training, DPMS', comb_full_vls))
    print(_summary_str('out of training, no DPMS', sing_full_vls))

    ax.spines[['right', 'top']].set_visible(False)
    # ax.set_xticks([-1, 0, 1])
    # ax.set_yticks([-1, 0, 1])

In [ ]:
vis_perf(eval_rs, ax, 'r_sq', min_vl=-1, max_plot_vl=1, title='R-Squared', ex_i=ex_s_i)
plt.savefig(save_folder / 'perf_r_sq.svg', format='svg', transparent=True)

In [ ]:
vis_perf(eval_rs, ax, 'corr', max_plot_vl=1, title='Correlation', ex_i=ex_s_i)
plt.savefig(save_folder / 'perf_corr.svg', format='svg', transparent=True)

In [ ]:
vis_perf(eval_rs, ax, 'elbo', min_vl=-600000.0, title='ELBO', ex_i=ex_s_i)
plt.savefig(save_folder / 'perf_elbo.svg', format='svg', transparent=True)